# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [1]:
!pip install -U transformers peft bitsandbytes accelerate datasets trl scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.8 MB/s eta 0:00:00


# 1. Authentication

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [3]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- LoRA ----
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])

    # ---- Training ----
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.1

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- BCE ----
    bce_weight: float = 1.0

    # ---- Prediction marker (plain text, NOT a special token) ----
    prediction_marker: str = "<|prediction|>"

    # ---- Paths ----
    output_dir: str = "./outputs"


cfg = ExperimentConfig()
print(f"Config: {cfg}")

Config: ExperimentConfig(model_name='Qwen/Qwen3-4B', max_seq_length=2048, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], num_train_epochs=3, per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.1, load_in_4bit=True, bnb_4bit_quant_type='nf4', use_double_quant=True, bce_weight=1.0, prediction_marker='<|prediction|>', output_dir='./outputs')


# 3. Load Training Data

In [4]:
import pandas as pd
from datasets import Dataset

train_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/train.csv")
eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/test.csv")

train_dataset = Dataset.from_pandas(train_data[["text"]].iloc[:500])
eval_dataset = Dataset.from_pandas(eval_data[["text"]])

print(f"Train: {len(train_dataset)} samples")
print(f"Eval:  {len(eval_dataset)} samples")

Train: 500 samples
Eval:  200 samples


# 4. Load Model + Tokenizer (QLoRA)
No special tokens are added. The prediction marker `<|prediction|>` is treated as
regular text and tokenized into multiple sub-tokens by the base tokenizer.
This means we do NOT need `resize_token_embeddings` or `modules_to_save`,
which keeps the trainable parameter count small.

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer (no special tokens added) ----
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ---- Encode the prediction marker as a multi-token sequence ----
prediction_marker_ids = tokenizer.encode(
    cfg.prediction_marker, add_special_tokens=False
)
print(f"Prediction marker '{cfg.prediction_marker}' -> token IDs: {prediction_marker_ids}")
print(f"  Decoded back: {[tokenizer.decode([t]) for t in prediction_marker_ids]}")

# ---- Get label token IDs for BCE loss ----
label_0_token_id = tokenizer.encode("0", add_special_tokens=False)[0]
label_1_token_id = tokenizer.encode("1", add_special_tokens=False)[0]

print(f"Token IDs -> '0': {label_0_token_id}, '1': {label_1_token_id}")
assert label_0_token_id != tokenizer.unk_token_id, "'0' mapped to UNK!"
assert label_1_token_id != tokenizer.unk_token_id, "'1' mapped to UNK!"

# ---- Load model (no embedding resize needed) ----
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16
)

print(f"Model loaded. Vocab size: {model.config.vocab_size}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded. Vocab size: 151936


# 5. Apply LoRA
Without `modules_to_save`, only the LoRA adapter weights are trainable.
The full `embed_tokens` and `lm_head` layers stay frozen.

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=cfg.lora_target_modules,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


# 6. BCE + SFT Dual-Loss Trainer
Since `<|prediction|>` is no longer a single special token, we search for its
multi-token subsequence in `input_ids` to locate where the model should predict 0/1.

In [ ]:
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss
from trl import SFTTrainer, SFTConfig


def find_subsequence(seq, subseq):
    """Find the starting index of subseq in seq. Returns -1 if not found."""
    subseq_len = len(subseq)
    for i in range(len(seq) - subseq_len + 1):
        if seq[i:i + subseq_len].tolist() == subseq:
            return i
    return -1


class BCESFTTrainer(SFTTrainer):
    """
    Extends SFTTrainer to add BCE loss on the <|prediction|> marker position.

    How it works:
    1. Standard causal LM loss is computed over all tokens (inherited).
    2. We find the multi-token subsequence for '<|prediction|>' in each sequence.
    3. The logit at the LAST token of that marker predicts the next token (0 or 1).
       We use that logit to compute BCE loss against the ground-truth label.
    4. Total loss = lm_loss + bce_weight * bce_loss
    """

    def __init__(self, *args, bce_weight=1.0, prediction_marker_ids=None,
                 label_0_token_id=None, label_1_token_id=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.bce_weight = bce_weight
        self.prediction_marker_ids = prediction_marker_ids
        self.label_0_token_id = label_0_token_id
        self.label_1_token_id = label_1_token_id
        self.bce_loss_fn = BCEWithLogitsLoss()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Override compute_loss to add BCE on <|prediction|> marker.

        In causal LM, the model predicts the NEXT token at each position.
        If the marker ends at position i, the logits at position i predict
        what comes at position i+1 (which is '0' or '1').
        """
        labels = inputs.get("labels")
        outputs = model(**inputs)
        lm_loss = outputs.loss
        logits = outputs.logits

        input_ids = inputs["input_ids"]
        batch_size = input_ids.size(0)
        marker_len = len(self.prediction_marker_ids)

        bce_losses = []
        for b in range(batch_size):
            start_idx = find_subsequence(input_ids[b], self.prediction_marker_ids)

            if start_idx == -1:
                continue

            # Position of the LAST token in the marker
            pred_pos = start_idx + marker_len - 1

            logit_0 = logits[b, pred_pos, self.label_0_token_id]
            logit_1 = logits[b, pred_pos, self.label_1_token_id]

            if pred_pos + 1 < input_ids.size(1):
                actual_next_token = input_ids[b, pred_pos + 1].item()
                if actual_next_token == self.label_1_token_id:
                    target = torch.tensor(1.0, device=logits.device)
                else:
                    target = torch.tensor(0.0, device=logits.device)

                bce_logit = logit_1 - logit_0
                bce_loss = self.bce_loss_fn(bce_logit, target)
                bce_losses.append(bce_loss)

        if bce_losses:
            bce_loss_mean = torch.stack(bce_losses).mean()
            total_loss = lm_loss + self.bce_weight * bce_loss_mean
        else:
            total_loss = lm_loss
            bce_loss_mean = torch.tensor(0.0, device=lm_loss.device)

        return (total_loss, outputs) if return_outputs else total_loss

# 7. Training

In [8]:
training_args = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    seed=42,
    report_to="none",
    max_length=cfg.max_seq_length,
    packing=False,
    dataset_text_field="text",
)

trainer = BCESFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    bce_weight=cfg.bce_weight,
    prediction_marker_ids=prediction_marker_ids,
    label_0_token_id=label_0_token_id,
    label_1_token_id=label_1_token_id,
)

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
10,24.928986
20,20.820287
30,18.314433
40,17.523257
50,15.643384
60,21.536226
70,13.802028
80,13.806300
90,13.600359
100,13.397461


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error The read operation timed out - silently ignoring the lookup for the file config.json in Qwen/Qwen3-4B.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in Qwen/Qwen3-4B - will assume that the vocabulary was not modified.
  warnings.warn(



Training complete!
Final train loss: 14.4327


# 8. Save LoRA Adapter

In [9]:
adapter_path = f"{cfg.output_dir}/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

import os
for f in os.listdir(adapter_path):
    size_mb = os.path.getsize(os.path.join(adapter_path, f)) / 1024 / 1024
    print(f"  {f}: {size_mb:.2f} MB")

import json
with open(os.path.join(adapter_path, "experiment_config.json"), "w") as f:
    json.dump(vars(cfg), f, indent=2)

Adapter saved to ./outputs/final_adapter
  tokenizer_config.json: 0.00 MB
  adapter_config.json: 0.00 MB
  README.md: 0.00 MB
  chat_template.jinja: 0.00 MB
  adapter_model.safetensors: 63.06 MB
  tokenizer.json: 10.89 MB
